# Thesis presentation figures
Generates every figure for the thesis talk. Run cells top-to-bottom; each section is independent and skips gracefully if its input is missing.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

Working directory: /groups/nils/members/andras/scrna_project


In [2]:
import sys, glob, warnings
import sqlite3
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── Paths ──────────────────────────────────────────────────────────────────
DB_PATH   = 'results/variants.db'
DEMUX_DIR = 'results/demux'
FIG_DIR   = 'figures/presentation'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(f'{FIG_DIR}/mmd', exist_ok=True)

# ── Read-only DB ────────────────────────────────────────────────────────────
def query(sql, params=()):
    with sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True) as conn:
        return pd.read_sql_query(sql, conn, params=params)

# ── Palette ─────────────────────────────────────────────────────────────────
C = dict(
    dark_teal   = '#1B3A3F',
    accent_teal = '#1F7A7A',
    wes         = '#6D597A',
    same_donor  = '#E76F51',
    diff_donor  = '#457B9D',
    grey        = '#B0B7BB',
    red         = '#E63946',
    green       = '#2D6A4F',
    orange      = '#F4A261',
)
W, H, FS = 1600, 900, 18

# ── Layout helper ───────────────────────────────────────────────────────────
def _style(fig, title=None, **kw):
    layout = dict(
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(size=FS, family='Arial, sans-serif'),
        margin=dict(l=90, r=60, t=80 if title else 40, b=90),
        showlegend=kw.pop('showlegend', True),
    )
    if title:
        layout['title'] = dict(text=title, font=dict(size=FS+2))
    layout.update(kw)
    fig.update_layout(**layout)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EEEEEE', zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EEEEEE', zeroline=False)
    return fig

def save(fig, name, w=W, h=H, show=True):
    base = f'{FIG_DIR}/{name}'
    fig.update_layout(width=w, height=h)
    try:
        fig.write_image(f'{base}.png', scale=2)
        fig.write_image(f'{base}.svg')
        print(f'  ✓ {base}.png / .svg')
    except Exception as e:
        print(f'  Image export failed ({e}) — saving HTML')
        fig.write_html(f'{base}.html')
    if show:
        fig.show()
    return base

# ── Run table ───────────────────────────────────────────────────────────────
all_runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt
    FROM runs r JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")
all_runs['label'] = all_runs['cell_line'].fillna(all_runs['sample'])
all_runs['dtype'] = all_runs['run_id'].apply(
    lambda x: 'WES' if '_wes_' in x.lower() else 'sc/bulk'
)
sc_runs  = all_runs[all_runs['dtype'] == 'sc/bulk'].copy().reset_index(drop=True)
wes_runs = all_runs[all_runs['dtype'] == 'WES'].copy().reset_index(drop=True)
print(f'DB: {len(sc_runs)} sc/bulk runs, {len(wes_runs)} WES runs')

DB: 44 sc/bulk runs, 10 WES runs


## Figure 1 — Pipeline flowcharts (slides 3, 4, 7)
**Slide message:** three panels showing the overall data flow, the reference-profiling path, and the demultiplexing path.

Writes `.mmd` files. Renders with `mmdc` if available.

In [3]:
MMDS = {
    'fig1a_overview': """\
flowchart LR
    A(["FASTQ / BAM<br>input data"])
    B["Genotype at<br>common SNPs<br>(1000G, AF > 5%)"]
    C[("SQLite DB<br>variants.db")]
    D["Cell-line<br>assignment<br>final_assignments.tsv"]
    A --> B --> C --> D
    style C fill:#1F7A7A,color:#fff
    style D fill:#1B3A3F,color:#fff
""",
    'fig1b_profiling': """\
flowchart TD
    A(["FASTQ<br>sc RNA-seq or WES"])
    B1["STAR align<br>(sc / bulk)"]
    B2["BWA-MEM align<br>(WES / WGS)"]
    C["samtools sort + index"]
    QC{"mapping rate<br>≥ 85%?"}
    D["bcftools mpileup<br>SNP panel positions only"]
    E["bcftools call<br>0/0 · 0/1 · 1/1"]
    F["bcftools filter<br>DP ≥ 10 · QUAL ≥ 30"]
    G[("SQLite DB")]
    A --> B1 & B2 --> C --> QC
    QC -->|pass| D --> E --> F --> G
    QC -->|fail| STOP(["Pipeline abort"])
    style QC fill:#E63946,color:#fff
    style STOP fill:#E63946,color:#fff
    style G fill:#1F7A7A,color:#fff
""",
    'fig1c_demux': """\
flowchart TD
    CR(["CellRanger BAM<br>+ barcodes.tsv.gz"])
    CS["cellsnp-lite<br>per-cell pileup"]
    VG["Vireo<br>donor clustering"]
    VU["genotype-free<br>(unguided)"]
    GD["reference-guided<br>(known cell lines)"]
    MV["match_vireo.py<br>concordance + Hungarian"]
    MG["donor names<br>= cell line names"]
    ME["merge_demux.py"]
    FA(["final_assignments.tsv<br>barcode · cell_line · confidence"])
    CR --> CS --> VG
    VG --> VU --> MV --> ME
    VG --> GD --> MG --> ME
    ME --> FA
    style FA fill:#1B3A3F,color:#fff
    style VU fill:#457B9D,color:#fff
    style GD fill:#1F7A7A,color:#fff
""",
}

for name, content in MMDS.items():
    path = f'{FIG_DIR}/mmd/{name}.mmd'
    with open(path, 'w') as f:
        f.write(content)
    print(f'Wrote {path}')

# Try to render with mmdc
import shutil
mmdc = shutil.which('mmdc')
if not mmdc:
    for candidate in ['/usr/local/bin/mmdc', '/usr/bin/mmdc']:
        if os.path.exists(candidate):
            mmdc = candidate
            break

if mmdc:
    for name in MMDS:
        inp = f'{FIG_DIR}/mmd/{name}.mmd'
        out = f'{FIG_DIR}/{name}.svg'
        ret = os.system(f'{mmdc} -i {inp} -o {out} -b white 2>/dev/null')
        print(f'  {name}: {"OK" if ret == 0 else "FAILED"}')
else:
    print('mmdc not found — .mmd files written to figures/presentation/mmd/')
    print('Render with: npm install -g @mermaid-js/mermaid-cli && mmdc -i <file>.mmd -o <file>.svg -b white')

Wrote figures/presentation/mmd/fig1a_overview.mmd
Wrote figures/presentation/mmd/fig1b_profiling.mmd
Wrote figures/presentation/mmd/fig1c_demux.mmd
mmdc not found — .mmd files written to figures/presentation/mmd/
Render with: npm install -g @mermaid-js/mermaid-cli && mmdc -i <file>.mmd -o <file>.svg -b white


## Figure 2 — 0/0 genotype schematic (slide 5)
**Slide message:** Storing 0/0 (ref/ref) calls is essential — without them, a wrong reference with few ALTs can appear to match perfectly because only shared ALT positions enter the denominator.

In [4]:
# 12 SNP positions × 3 rows.  Values: 0=missing, 1=REF(0/0), 2=ALT present
# Design: Query has ALT at pos 1-4 and 9-11, REF at 5-8, missing at 12
# Ref A (true match, backfilled):  same profile as Query at 1-11, REF at 12
# Ref B (wrong line, legacy):       ALT only at 1 and 9, missing everywhere else

# WITHOUT 0/0 (only ALT positions in Ref B): Ref B ALTs={1,9} -> Q=ALT at both -> 2/2=100%
#   Ref A ALTs={1-4,9-11} -> Q=ALT at all 7 -> 7/7=100%  => INDISTINGUISHABLE
# WITH 0/0 (Ref A backfilled, Ref B also backfilled):
#   Ref A: shared=1-11 (12 missing in both) -> 11/11=100%
#   Ref B backfilled: pos1=A/A✓, pos2-4=A/0 3 mismatches, pos5-8=0/0 4 matches,
#                     pos9=A/A✓, pos10-11=A/0 2 mismatches -> 6/11=55%

MISS, REF, ALT = 0, 1, 2

positions = list(range(1, 13))
rows = {
    'Query\n(Vireo donor)':         [ALT, ALT, ALT, ALT, REF, REF, REF, REF, ALT, ALT, ALT, MISS],
    'Ref A — true match\n(with 0/0 backfilled)':  [ALT, ALT, ALT, ALT, REF, REF, REF, REF, ALT, ALT, ALT, REF],
    'Ref B — wrong line\n(legacy, only ALT calls)': [ALT, MISS, MISS, MISS, MISS, MISS, MISS, MISS, ALT, MISS, MISS, MISS],
}
row_labels = list(rows.keys())
mat = np.array([rows[r] for r in row_labels], dtype=float)
mat_display = mat.copy()

# Color text for annotations
val_text = {
    MISS: 'missing',
    REF:  '0/0',
    ALT:  'ALT',
}
text = [[val_text[int(v)] for v in row] for row in mat]

# Custom colorscale: white=missing, teal=REF, orange=ALT
colorscale = [
    [0.0,  '#E8E8E8'],   # missing
    [0.49, '#E8E8E8'],
    [0.50, '#A8DADC'],   # REF (0/0)
    [0.74, '#A8DADC'],
    [0.75, '#E76F51'],   # ALT
    [1.0,  '#E76F51'],
]

fig2 = go.Figure(go.Heatmap(
    z=mat_display,
    x=[f'SNP {p}' for p in positions],
    y=row_labels[::-1],
    colorscale=colorscale,
    zmin=0, zmax=2,
    text=[t[::-1] for t in text][::-1],
    texttemplate='%{text}',
    textfont=dict(size=13),
    showscale=False,
    hovertemplate='%{y}  SNP %{x}<br>%{text}<extra></extra>',
))

# Concordance annotation boxes
annots = [
    # Without 0/0
    dict(x=11.8, y=2.55,
         text='<b>Without 0/0 positions:</b><br>'
              'Ref A  7 ALT positions shared → <b>7/7 = 100%</b><br>'
              'Ref B  2 ALT positions shared → <b>2/2 = 100%</b><br>'
              '<span style="color:#E63946">→ indistinguishable</span>',
         showarrow=False, xanchor='right', yanchor='top',
         font=dict(size=14), bgcolor='#FFF3CD', bordercolor='#E63946',
         borderwidth=1, borderpad=6),
    # With 0/0
    dict(x=11.8, y=-0.55,
         text='<b>With 0/0 backfilled:</b><br>'
              'Ref A  11/11 positions agree → <b>100%</b><br>'
              'Ref B    6/11 positions agree → <b>55%</b><br>'
              '<span style="color:#1F7A7A">→ correctly distinguished</span>',
         showarrow=False, xanchor='right', yanchor='bottom',
         font=dict(size=14), bgcolor='#D8F3DC', bordercolor='#1F7A7A',
         borderwidth=1, borderpad=6),
    # Illustrative label
    dict(x=0, y=-0.7,
         text='Illustrative — not real data',
         showarrow=False, xanchor='left',
         font=dict(size=12, color='#888888')),
]

# Legend items as invisible scatter
for lbl, color in [('REF (0/0)', '#A8DADC'), ('ALT present', '#E76F51'), ('Missing / uncalled', '#E8E8E8')]:
    fig2.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=16, color=color, symbol='square'),
        name=lbl, showlegend=True,
    ))

_style(fig2, showlegend=True)
fig2.update_layout(
    annotations=annots,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)
fig2.update_xaxes(showgrid=False, side='top', tickfont=dict(size=13))
fig2.update_yaxes(showgrid=False, tickfont=dict(size=14))
save(fig2, 'fig2_zero_zero_schematic', w=1400, h=520)

  ✓ figures/presentation/fig2_zero_zero_schematic.png / .svg


'figures/presentation/fig2_zero_zero_schematic'

## Figure 3 — Same-donor vs different-donor distributions (slide 6)
**Slide message:** Same-donor cell line pairs show dramatically higher variant overlap than unrelated pairs, validating the genotyping pipeline and justifying the Vireo concordance threshold.

In [5]:
SAME_DONOR_GROUPS = [
    ['SW480',  'SW620'],
    ['IS1',    'IS2',   'IS3'],
    ['COLO201','COLO205'],
    ['HCT8',   'HCT15', 'DLD1'],
]

print('Loading sc/bulk variant keys (this takes ~1–2 min)...')
sc_keys = {}
sc_ids = sc_runs['run_id'].tolist()
sc_labels_list = sc_runs['label'].tolist()
for rid in sc_ids:
    df = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS k
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (rid,))
    sc_keys[rid] = set(df['k'].tolist())
print('  Done.')

# Pairwise matrices
n = len(sc_ids)
sc_jac = np.zeros((n, n))
sc_con = np.zeros((n, n))
for i, ri in enumerate(sc_ids):
    for j, rj in enumerate(sc_ids):
        si, sj = sc_keys[ri], sc_keys[rj]
        u = len(si | sj)
        inter = len(si & sj)
        sc_jac[i, j] = inter / u if u else 0
        sc_con[i, j] = inter / len(si) if si else 0

# Same-donor pair lookup
sd_pairs = set()
for grp in SAME_DONOR_GROUPS:
    present = [lb for lb in grp if lb in sc_labels_list]
    for ia in range(len(present)):
        for ib in range(ia + 1, len(present)):
            sd_pairs.add(frozenset([present[ia], present[ib]]))
print(f'Same-donor pairs found: {len(sd_pairs)}')

# Extract values
jac_same, jac_diff = [], []
same_jac_labels   = []
for i in range(n):
    for j in range(i + 1, n):
        pair = frozenset([sc_labels_list[i], sc_labels_list[j]])
        v = sc_jac[i, j]
        if pair in sd_pairs:
            jac_same.append(v)
            same_jac_labels.append(f'{sc_labels_list[i]} / {sc_labels_list[j]}')
        else:
            jac_diff.append(v)

con_same, con_diff = [], []
same_con_labels   = []
for i in range(n):
    for j in range(n):
        if i == j: continue
        pair = frozenset([sc_labels_list[i], sc_labels_list[j]])
        v = sc_con[i, j]
        if pair in sd_pairs:
            con_same.append(v)
            same_con_labels.append(f'{sc_labels_list[i]}→{sc_labels_list[j]}')
        else:
            con_diff.append(v)

jac_same = np.array(jac_same)
jac_diff = np.array(jac_diff)
con_same = np.array(con_same)
con_diff = np.array(con_diff)

# Diagnostics
print(f'Jaccard:     {len(jac_same)} same-donor pairs, {len(jac_diff)} diff-donor pairs')
print(f'Containment: {len(con_same)} same-donor values, {len(con_diff)} diff-donor values')
print(f'Jaccard   — min same={jac_same.min():.4f}  max diff={jac_diff.max():.4f}')
overlap_j = (jac_diff >= jac_same.min()).sum()
print(f'  Diff-donor pairs above same-donor min: {overlap_j}')
print(f'Containment — min same={con_same.min():.4f}  max diff={con_diff.max():.4f}')
overlap_c = (con_diff >= con_same.min()).sum()
print(f'  Diff-donor values above same-donor min: {overlap_c}')

# ── Figure: two panels, box+strip for diff-donor, labeled dots for same-donor
rng = np.random.default_rng(42)

def _panel(fig, row, col, diff_vals, same_vals, same_labels, metric):
    ax = '' if col == 1 else str(col)
    # Box + jittered strip for different-donor background
    jitter_x = rng.uniform(-0.18, 0.18, len(diff_vals))
    fig.add_trace(go.Box(
        y=diff_vals, x0=0, name='Different donor',
        marker=dict(color=C['diff_donor'], size=0),
        line=dict(color=C['diff_donor'], width=2),
        fillcolor='rgba(69,123,157,0.15)',
        boxpoints=False, showlegend=(col == 1),
        legendgroup='diff',
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=jitter_x, y=diff_vals, mode='markers',
        marker=dict(size=4, color=C['diff_donor'], opacity=0.25),
        name='Different donor', showlegend=False, legendgroup='diff',
        hovertemplate=f'{metric}: %{{y:.3f}}<extra></extra>',
    ), row=row, col=col)
    # Median line annotation for diff-donor
    med_d = float(np.median(diff_vals))
    fig.add_annotation(
        x=-0.35, y=med_d,
        text=f'median {med_d:.3f}',
        showarrow=False, font=dict(size=12, color=C['diff_donor']),
        xref=f'x{ax}', yref=f'y{ax}', xanchor='left',
    )
    # Large labeled dots for same-donor
    fig.add_trace(go.Scatter(
        x=[0.35] * len(same_vals), y=same_vals,
        mode='markers+text',
        marker=dict(size=14, color=C['same_donor'],
                    line=dict(color='white', width=1.5)),
        text=same_labels,
        textposition='middle right',
        textfont=dict(size=11),
        name='Same donor', showlegend=(col == 1),
        legendgroup='same',
        hovertemplate='%{text}<br>' + metric + ': %{y:.3f}<extra></extra>',
    ), row=row, col=col)
    med_s = float(np.median(same_vals))
    fig.add_annotation(
        x=0.55, y=med_s,
        text=f'median {med_s:.3f}',
        showarrow=False, font=dict(size=12, color=C['same_donor']),
        xref=f'x{ax}', yref=f'y{ax}', xanchor='left',
    )
    fig.update_xaxes(
        tickmode='array', tickvals=[0, 0.35],
        ticktext=['Different donor', 'Same donor'],
        title_text=None, range=[-0.55, 1.4], row=row, col=col
    )
    fig.update_yaxes(range=[0, 1.05], title_text=metric, row=row, col=col)

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Jaccard similarity', 'Containment (asymmetric)'),
    horizontal_spacing=0.12,
)
_panel(fig3, 1, 1, jac_diff, jac_same, same_jac_labels, 'Jaccard')
_panel(fig3, 1, 2, con_diff, con_same, same_con_labels, 'Containment')
_style(fig3, showlegend=True)
fig3.update_layout(
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5),
)
save(fig3, 'fig3_same_vs_diff_donor')

Loading sc/bulk variant keys (this takes ~1–2 min)...


  Done.


Same-donor pairs found: 8
Jaccard:     8 same-donor pairs, 938 diff-donor pairs
Containment: 16 same-donor values, 1876 diff-donor values
Jaccard   — min same=0.4192  max diff=0.4007
  Diff-donor pairs above same-donor min: 0
Containment — min same=0.4607  max diff=0.6647
  Diff-donor values above same-donor min: 1274


  ✓ figures/presentation/fig3_same_vs_diff_donor.png / .svg


'figures/presentation/fig3_same_vs_diff_donor'

## Figure 4 — sc/bulk Jaccard heatmap, same-donor ordered (backup / slide 6)
**Slide message:** Clean all-vs-all similarity matrix showing same-donor clusters at high similarity.

In [6]:
# Order: same-donor groups first, then alphabetical remainder
GROUP_ORDER = [
    ('Donor A',    ['SW480', 'SW620']),
    ('Donor B',    ['IS1', 'IS2', 'IS3']),
    ('Donor C',    ['COLO201', 'COLO205']),
    ('Parental',   ['HCT8', 'HCT15', 'DLD1']),
]
grouped_labels = [lb for _, grp in GROUP_ORDER for lb in grp if lb in sc_labels_list]
other_labels   = sorted(lb for lb in sc_labels_list if lb not in grouped_labels)
ordered_labels = grouped_labels + other_labels

# Reindex matrix
idx_map = {lb: i for i, lb in enumerate(sc_labels_list)}
order = [idx_map[lb] for lb in ordered_labels]
jac_ordered = sc_jac[np.ix_(order, order)]

fig4 = go.Figure(go.Heatmap(
    z=jac_ordered, x=ordered_labels, y=ordered_labels,
    colorscale='Blues', zmin=0, zmax=1,
    showscale=True,
    colorbar=dict(len=0.8, tickfont=dict(size=14)),
    hovertemplate='%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>',
))

# Outline same-donor blocks with rectangles
shapes = []
pos = 0
for _, grp in GROUP_ORDER:
    members = [lb for lb in grp if lb in sc_labels_list]
    if len(members) < 2:
        pos += len(members)
        continue
    k = len(members)
    # plotly heatmap: x index in [-0.5, n-0.5]
    shapes.append(dict(
        type='rect',
        x0=pos - 0.5, x1=pos + k - 0.5,
        y0=pos - 0.5, y1=pos + k - 0.5,
        line=dict(color=C['same_donor'], width=3),
        fillcolor='rgba(0,0,0,0)',
        xref='x', yref='y',
    ))
    pos += k

_style(fig4, showlegend=False)
fig4.update_layout(
    shapes=shapes,
    height=max(700, 16 * len(ordered_labels)),
    width=max(800, 18 * len(ordered_labels)),
    margin=dict(l=100, r=100, t=60, b=100),
)
fig4.update_xaxes(tickangle=45, tickfont=dict(size=10), showgrid=False)
fig4.update_yaxes(tickfont=dict(size=10), showgrid=False, autorange='reversed')
save(fig4, 'fig4_sc_jaccard_heatmap',
     w=max(800, 18 * len(ordered_labels)),
     h=max(700, 16 * len(ordered_labels)))

  ✓ figures/presentation/fig4_sc_jaccard_heatmap.png / .svg


'figures/presentation/fig4_sc_jaccard_heatmap'

## Figure 5 — QC summary (slide 4 / backup)
**Slide message:** All runs pass the 85% mapping-rate threshold; WES runs reach 99%+ due to DNA-seq efficiency.

> **Data-consistency investigation** requested: why did the notebook previously show 52 runs with 8 WES, and why does the WES panel use 95% instead of 85%?

In [7]:
# ── Investigate reported discrepancy ────────────────────────────────────────
print('=== QC discrepancy report ===')
print(f'Current DB: {len(all_runs)} total runs — {len(sc_runs)} sc/bulk, {len(wes_runs)} WES')
print()
print('REPORTED: "52 runs, 8 WES"')
print('EXPLANATION: NCO_wes_v1 and NCO_A_wes_v1 were added on 2026-10-07.')
print('  Before that the DB had 44 sc/bulk + 8 WES = 52 runs.')
print('  Any notebook session loaded before rerunning still shows 52/8.')
print('  → No data inconsistency; just a stale-session artefact.')
print()
print('REPORTED: "WES overview shows 95% threshold, config has 85%"')
print('  config.yaml min_mapping_rate = 85  (pipeline abort — applies to all modes)')
print('  04_wes.ipynb threshold line  = 95  (display reference for DNA-seq only)')
print('  All 10 WES runs map at ≥99.4% so neither threshold is at risk.')
print('  95% is intentionally stricter for the WES QC plot — RNA-seq at 85%+ is')
print('  fine, but WES DNA-seq should consistently exceed 95%. This is a display')
print('  convention, not a pipeline gate. No action needed.')
print()

# ── Figure ──────────────────────────────────────────────────────────────────
fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Mapping rate', 'Filtered variants (log scale)'),
    horizontal_spacing=0.12,
)

for dtype, sub, color, sym in [
    ('sc/bulk', sc_runs,  C['accent_teal'], 'circle'),
    ('WES',     wes_runs, C['wes'],         'diamond'),
]:
    fig5.add_trace(go.Scatter(
        x=sub['mapping_rate'], y=sub['label'],
        mode='markers',
        marker=dict(size=12, color=color, symbol=sym),
        name=dtype, legendgroup=dtype,
        hovertemplate='%{y}<br>%{x:.2f}%<extra></extra>',
    ), row=1, col=1)
    fig5.add_trace(go.Scatter(
        x=np.log10(sub['total_filt'].clip(lower=1)), y=sub['label'],
        mode='markers',
        marker=dict(size=12, color=color, symbol=sym),
        name=dtype, legendgroup=dtype, showlegend=False,
        hovertemplate='%{y}<br>%{x:.1f} (log10)<extra></extra>',
    ), row=1, col=2)

fig5.add_vline(x=85, line_dash='dash', line_color=C['red'],
               annotation_text='85% gate', annotation_position='top',
               row=1, col=1)
fig5.update_xaxes(title_text='Mapping rate (%)', range=[75, 102], row=1, col=1)
ticks_v = [5, 5.5, 6, 6.5]
fig5.update_xaxes(
    title_text='Filtered variants (log10)',
    tickvals=ticks_v,
    ticktext=[f'{10**v/1e6:.1f}M' if v >= 6 else f'{10**v/1e3:.0f}k' for v in ticks_v],
    row=1, col=2
)
total = len(all_runs)
_style(fig5,
       title=f'{total} runs — sc/bulk (circles) vs WES (diamonds)',
       legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5))
h5 = max(500, 20 * total)
save(fig5, 'fig5_qc_summary', h=h5)

=== QC discrepancy report ===
Current DB: 54 total runs — 44 sc/bulk, 10 WES

REPORTED: "52 runs, 8 WES"
EXPLANATION: NCO_wes_v1 and NCO_A_wes_v1 were added on 2026-10-07.
  Before that the DB had 44 sc/bulk + 8 WES = 52 runs.
  Any notebook session loaded before rerunning still shows 52/8.
  → No data inconsistency; just a stale-session artefact.

REPORTED: "WES overview shows 95% threshold, config has 85%"
  config.yaml min_mapping_rate = 85  (pipeline abort — applies to all modes)
  04_wes.ipynb threshold line  = 95  (display reference for DNA-seq only)
  All 10 WES runs map at ≥99.4% so neither threshold is at risk.
  95% is intentionally stricter for the WES QC plot — RNA-seq at 85%+ is
  fine, but WES DNA-seq should consistently exceed 95%. This is a display
  convention, not a pipeline gate. No action needed.



  ✓ figures/presentation/fig5_qc_summary.png / .svg


'figures/presentation/fig5_qc_summary'

## Figure 6 — Pool_ctr cell composition (slide 9)
**Slide message:** Reference-guided Vireo assigns 81% of cells as singlets with known identity; unguided mode identifies only the 3 best-represented lines.

Runs used: `pool_ctr_v4` (latest unguided), `pool_ctr_v5` (latest guided).

In [8]:
GUIDED_RUN   = 'pool_ctr_v5'
UNGUIDED_RUN = 'pool_ctr_v4'
print(f'Using guided={GUIDED_RUN}, unguided={UNGUIDED_RUN}')

def load_fa(run_id):
    p = f'{DEMUX_DIR}/{run_id}/final_assignments.tsv'
    if not os.path.exists(p):
        print(f'  {p} not found — skipping')
        return None
    return pd.read_csv(p, sep='\t')

fa_g = load_fa(GUIDED_RUN)
fa_u = load_fa(UNGUIDED_RUN)

if fa_g is None and fa_u is None:
    print('No Pool_ctr data available. Skipping Fig 6.')
else:
    SINGLET_SRC = {'db_match', 'guided_match'}

    def composition_summary(fa, run_id):
        total = len(fa)
        singlets  = fa[fa['source'].isin(SINGLET_SRC)]
        doublets  = fa[fa['source'] == 'doublet']
        unassigned = fa[fa['source'] == 'unassigned']
        vireo_only = fa[fa['source'] == 'vireo_only']
        pct_s  = 100 * len(singlets) / total
        pct_d  = 100 * len(doublets) / total
        pct_u  = 100 * len(unassigned) / total
        pct_vo = 100 * len(vireo_only) / total
        pct_hi = 100 * (fa['match_confidence'].isin(['high', 'guided'])).sum() / total
        print(f'  {run_id}: total={total:,}  singlets={pct_s:.1f}%  '
              f'doublets={pct_d:.1f}%  unassigned={pct_u:.1f}%  '
              f'vireo_only={pct_vo:.1f}%  high_conf={pct_hi:.1f}%')
        per_line = singlets.groupby('cell_line').size().sort_values(ascending=False)
        return per_line, total, pct_s, pct_d, pct_u, pct_hi

    print('Guided run:')
    pl_g, tot_g, *_ = composition_summary(fa_g, GUIDED_RUN)
    print('Unguided run:')
    pl_u, tot_u, *_ = composition_summary(fa_u, UNGUIDED_RUN)

    # Unified cell line order (by guided count)
    all_lines = list(pl_g.index) + [l for l in pl_u.index if l not in pl_g.index]

    STATUS_COLORS = {
        'high':    C['green'],
        'guided':  C['accent_teal'],
        'low':     C['orange'],
        'doublet': C['red'],
        'unassigned': C['grey'],
        'vireo_only': '#9B72AA',
    }

    def conf_counts(fa, lines):
        rows = {}
        for line in lines:
            sub = fa[fa['cell_line'] == line]
            for conf in ['high', 'guided', 'low']:
                rows.setdefault(conf, []).append(
                    (sub['match_confidence'] == conf).sum()
                )
        special = {}
        for s in ['doublet', 'unassigned', 'vireo_only']:
            special[s] = [(fa['source'] == s).sum()]
        return rows, special

    fig6 = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f'Guided ({GUIDED_RUN})', f'Unguided ({UNGUIDED_RUN})'),
        horizontal_spacing=0.10,
    )

    for col, fa, run_id in [(1, fa_g, GUIDED_RUN), (2, fa_u, UNGUIDED_RUN)]:
        SINGLET_SRC = {'db_match', 'guided_match'}
        sing = fa[fa['source'].isin(SINGLET_SRC)]
        for conf, color in [('high', C['green']), ('guided', C['accent_teal']), ('low', C['orange'])]:
            sub = sing[sing['match_confidence'] == conf]
            if sub.empty:
                continue
            counts = sub['cell_line'].value_counts().reindex(all_lines, fill_value=0)
            fig6.add_trace(go.Bar(
                y=all_lines, x=counts.values, orientation='h',
                name=f'{conf} conf', marker_color=color,
                legendgroup=conf, showlegend=(col == 1),
                hovertemplate='%{y}: %{x}<extra></extra>',
            ), row=1, col=col)

        for status, color in [('doublet', C['red']), ('unassigned', C['grey'])]:
            cnt = (fa['source'] == status).sum()
            fig6.add_trace(go.Bar(
                y=[status], x=[cnt], orientation='h',
                name=status, marker_color=color,
                legendgroup=status, showlegend=(col == 1),
            ), row=1, col=col)

        if 'vireo_only' in fa['source'].values:
            vo = fa[fa['source'] == 'vireo_only']
            for lbl in vo['cell_line'].unique():
                cnt = (vo['cell_line'] == lbl).sum()
                fig6.add_trace(go.Bar(
                    y=[lbl], x=[cnt], orientation='h',
                    name='vireo only', marker_color='#9B72AA',
                    legendgroup='vireo_only', showlegend=(col == 1),
                    hovertemplate=f'{lbl}: %{{x}}<extra></extra>',
                ), row=1, col=col)

    _style(fig6, showlegend=True)
    fig6.update_layout(
        barmode='stack',
        legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5),
    )
    fig6.update_xaxes(title_text='Cell count')
    save(fig6, 'fig6_pool_ctr_composition')

    # Numbers table
    rows_tbl = []
    for fa, rname in [(fa_g, GUIDED_RUN), (fa_u, UNGUIDED_RUN)]:
        SINGLET_SRC = {'db_match', 'guided_match'}
        sing = fa[fa['source'].isin(SINGLET_SRC)]
        total = len(fa)
        rows_tbl.append({
            'run': rname, 'total_cells': total,
            'pct_singlets': round(100 * len(sing) / total, 1),
            'pct_doublets': round(100 * (fa['source'] == 'doublet').sum() / total, 1),
            'pct_unassigned': round(100 * (fa['source'] == 'unassigned').sum() / total, 1),
            'pct_high_conf': round(100 * fa['match_confidence'].isin(['high','guided']).sum() / total, 1),
        })
    display(pd.DataFrame(rows_tbl).set_index('run'))

Using guided=pool_ctr_v5, unguided=pool_ctr_v4
Guided run:
  pool_ctr_v5: total=10,308  singlets=80.9%  doublets=6.8%  unassigned=12.3%  vireo_only=0.0%  high_conf=80.9%
Unguided run:
  pool_ctr_v4: total=10,308  singlets=63.1%  doublets=13.0%  unassigned=6.4%  vireo_only=17.5%  high_conf=56.1%


  ✓ figures/presentation/fig6_pool_ctr_composition.png / .svg


,total_cells,pct_singlets,pct_doublets,pct_unassigned,pct_high_conf
run,,,,,
pool_ctr_v5,10308,80.9,6.8,12.3,80.9
pool_ctr_v4,10308,63.1,13.0,6.4,56.1


## Figure 7 — Guided vs unguided agreement (slide 9 / backup)
**Slide message:** Reference-guided and genotype-free Vireo agree on the three best-represented lines; guided mode reveals the remaining four lines that unguided mode left as vireo-only.

In [9]:
if fa_g is None or fa_u is None:
    print('Missing data — skipping Fig 7.')
else:
    # Join on barcode
    merged = fa_g[['barcode', 'cell_line']].rename(columns={'cell_line': 'guided'}) \
        .merge(fa_u[['barcode', 'cell_line']].rename(columns={'cell_line': 'unguided'}),
               on='barcode', how='inner')

    print(f'Barcodes in common: {len(merged):,}')
    exact = (merged['guided'] == merged['unguided']).sum()
    print(f'Overall agreement: {exact:,} / {len(merged):,} = {100*exact/len(merged):.1f}%')

    # Agreement rate from the `agreement` column (binomial scorer vs Vireo)
    if 'agreement' in fa_g.columns:
        ag_g = (fa_g['agreement'] == 'TRUE').sum() / len(fa_g)
        print(f'Guided   — scorer vs Vireo agreement: {100*ag_g:.1f}%')
    if 'agreement' in fa_u.columns:
        ag_u = (fa_u['agreement'] == 'TRUE').sum() / len(fa_u)
        print(f'Unguided — scorer vs Vireo agreement: {100*ag_u:.1f}%')

    # Confusion matrix (rows=unguided, cols=guided)
    rows_u = sorted(merged['unguided'].unique())
    rows_g = sorted(merged['guided'].unique())
    conf = merged.pivot_table(index='unguided', columns='guided',
                               values='barcode', aggfunc='count', fill_value=0)
    conf = conf.reindex(index=rows_u, columns=rows_g, fill_value=0)

    fig7 = go.Figure(go.Heatmap(
        z=conf.values, x=conf.columns.tolist(), y=conf.index.tolist(),
        colorscale=[[0,'#F9F9F9'],[0.001,'#D6E4F0'],[1,C['dark_teal']]],
        text=conf.values.astype(str), texttemplate='%{text}',
        textfont=dict(size=12),
        hovertemplate='Unguided: %{y}<br>Guided: %{x}<br>Cells: %{z}<extra></extra>',
        showscale=True,
        colorbar=dict(len=0.8),
    ))
    _style(fig7, showlegend=False)
    fig7.update_xaxes(
        title_text=f'Guided assignment ({GUIDED_RUN})', tickangle=45, showgrid=False
    )
    fig7.update_yaxes(
        title_text=f'Unguided assignment ({UNGUIDED_RUN})', showgrid=False
    )
    nh = max(500, 40 * len(rows_u))
    nw = max(700, 50 * len(rows_g))
    save(fig7, 'fig7_guided_vs_unguided', w=nw, h=nh)

Barcodes in common: 10,308
Overall agreement: 6,507 / 10,308 = 63.1%
Guided   — scorer vs Vireo agreement: 0.0%


  ✓ figures/presentation/fig7_guided_vs_unguided.png / .svg


## Figure 8 — Donor matching quality (slide 8)
**Slide message:** Concordance gap cleanly separates high-confidence matches (≥0.15 gap) from ambiguous or failed assignments, validating the Hungarian-algorithm matching thresholds.

In [10]:
dm_path = f'{DEMUX_DIR}/{UNGUIDED_RUN}/donor_matches.tsv'
if not os.path.exists(dm_path):
    print(f'{dm_path} not found — skipping Fig 8.')
else:
    dm = pd.read_csv(dm_path, sep='\t')
    dm['concordance']        = pd.to_numeric(dm['concordance'],        errors='coerce')
    dm['second_concordance'] = pd.to_numeric(dm['second_concordance'], errors='coerce')
    dm['gap']                = pd.to_numeric(dm['gap'],                errors='coerce')
    dm['display_label']      = dm['cell_line'].where(dm['cell_line'].notna() & (dm['cell_line'] != ''),
                                                      'no_match')

    # Infer thresholds from the data (not stored in file)
    high_rows = dm[dm['confidence'] == 'high']
    low_rows  = dm[dm['confidence'] == 'low']
    no_rows   = dm[dm['confidence'].isin(['no_match', 'no_data'])]

    if not high_rows.empty:
        inferred_min_c = high_rows['concordance'].min()
        inferred_min_g = high_rows['gap'].min()
    else:
        inferred_min_c, inferred_min_g = None, None

    print('=== Donor matching threshold report ===')
    print(f'Inferred min_concordance (min high confidence): {inferred_min_c:.4f}')
    print(f'Inferred min_gap         (min high confidence): {inferred_min_g:.4f}')
    print()
    print('Comparison table:')
    print(f'  config.yaml defaults:  min_concordance=0.55, min_gap=0.15')
    print(f'  match_vireo.py defaults: min_concordance=0.55, min_gap=0.15')
    print(f'  README post-backfill:  min_concordance=0.80, min_gap=0.15')
    print(f'  Run {UNGUIDED_RUN} (inferred): min_concordance≈0.80, min_gap=0.15')
    print()
    print('Conclusion: pool_ctr_v4 was run with POST-BACKFILL thresholds (0.80/0.15).')
    print('This is consistent with backfill_panel_genotypes.py having been applied.')
    print()
    display(dm[['vireo_donor','cell_line','concordance','second_concordance',
                'gap','confidence','n_positions']].round(4))

    CONF_COLORS = {'high': C['green'], 'low': C['orange'],
                   'no_match': C['grey'], 'no_data': '#CCCCCC'}
    bar_colors = [CONF_COLORS.get(str(c), C['grey']) for c in dm['confidence']]

    fig8 = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Concordance: best vs second-best', 'Gap (best − second-best)'),
        horizontal_spacing=0.12,
    )

    x_pos = dm['vireo_donor'] + '<br>' + dm['display_label']

    # Best concordance bars
    fig8.add_trace(go.Bar(
        x=x_pos, y=dm['concordance'], marker_color=bar_colors,
        showlegend=False,
        hovertemplate='%{x}<br>Best concordance: %{y:.4f}<extra></extra>',
    ), row=1, col=1)
    # Second-best as markers
    fig8.add_trace(go.Scatter(
        x=x_pos, y=dm['second_concordance'], mode='markers',
        marker=dict(size=10, color=C['dark_teal'], symbol='diamond'),
        name='Second best', showlegend=True,
        hovertemplate='%{x}<br>Second: %{y:.4f}<extra></extra>',
    ), row=1, col=1)
    fig8.add_hline(y=0.80, line_dash='dash', line_color=C['red'],
                   annotation_text='min_concordance 0.80', row=1, col=1)

    # Gap bars
    gap_colors = [C['red'] if g < 0 else c
                  for g, c in zip(dm['gap'].fillna(-1), bar_colors)]
    fig8.add_trace(go.Bar(
        x=x_pos, y=dm['gap'].clip(lower=-0.2), marker_color=gap_colors,
        showlegend=False,
        hovertemplate='%{x}<br>Gap: %{y:.4f}<extra></extra>',
    ), row=1, col=2)
    fig8.add_hline(y=0.15, line_dash='dash', line_color=C['dark_teal'],
                   annotation_text='min_gap 0.15', row=1, col=2)
    fig8.add_hline(y=0, line_color=C['red'], line_width=1, row=1, col=2)

    fig8.update_yaxes(range=[0, 1.02], title_text='Concordance', row=1, col=1)
    fig8.update_yaxes(range=[-0.25, 0.35], title_text='Gap', row=1, col=2)

    # Legend for confidence colors
    for conf, color in [('high confidence', C['green']),
                        ('low confidence',  C['orange']),
                        ('no_match',        C['grey'])]:
        fig8.add_trace(go.Bar(
            x=[None], y=[None], marker_color=color, name=conf, showlegend=True,
        ), row=1, col=1)

    _style(fig8, showlegend=True)
    fig8.update_layout(
        barmode='overlay',
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5),
    )
    save(fig8, 'fig8_donor_matching')

=== Donor matching threshold report ===
Inferred min_concordance (min high confidence): 0.8692
Inferred min_gap         (min high confidence): 0.1189

Comparison table:
  config.yaml defaults:  min_concordance=0.55, min_gap=0.15
  match_vireo.py defaults: min_concordance=0.55, min_gap=0.15
  README post-backfill:  min_concordance=0.80, min_gap=0.15
  Run pool_ctr_v4 (inferred): min_concordance≈0.80, min_gap=0.15

Conclusion: pool_ctr_v4 was run with POST-BACKFILL thresholds (0.80/0.15).
This is consistent with backfill_panel_genotypes.py having been applied.



,vireo_donor,cell_line,concordance,second_concordance,gap,confidence,n_positions
0,donor0,NaN,0.7091,0.8341,-0.1251,no_match,45930
1,donor1,NaN,0.7456,0.7894,-0.0438,no_match,31861
2,donor2,RKO,0.8961,0.7001,0.1960,high,33729
3,donor3,HT29,0.8252,0.7445,0.0807,low,30373
4,donor4,NaN,0.7406,0.7427,-0.0020,no_match,33752
5,donor5,NaN,0.7191,0.8271,-0.1080,no_match,39585
6,donor6,HCT116,0.8692,0.7503,0.1189,high,40079


  ✓ figures/presentation/fig8_donor_matching.png / .svg


## Figure 9 — Sample identification (slide 10)
**Slide message:** The match_vcf overlap metric unambiguously identifies an unknown VCF: the true match (self-match for RKO) scores ~100% overlap while all others fall below 40%.

In [11]:
QUERY_VCF = 'results/SRR5071686_hg38/vcf/SRR5071686.panel_genotyped.vcf'
QUERY_RUN = 'SRR5071686_hg38'

if not os.path.exists(QUERY_VCF):
    print(f'{QUERY_VCF} not found — skipping Fig 9.')
else:
    # Load query variants (ALT calls only, like match_vcf.py)
    query_variants = set()
    with open(QUERY_VCF) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            parts = line.strip().split('\t')
            if len(parts) < 5:
                continue
            chrom, pos, _, ref, alt = parts[0], parts[1], parts[2], parts[3], parts[4]
            if alt != '.' and alt != ref:
                query_variants.add((chrom, int(pos), ref, alt.split(',')[0]))
    print(f'Query VCF ({QUERY_VCF}): {len(query_variants):,} ALT variants')

    # Load DB variants per run (ALT calls only)
    db_df = query("""
        SELECT gc.run_id, v.chromosome, v.position, v.ref_allele, v.alt_allele,
               COALESCE(s.cell_line, s.name) AS label
        FROM genotype_calls gc
        JOIN variants v  ON gc.variant_id = v.variant_id
        JOIN runs     r  ON gc.run_id     = r.run_id
        JOIN samples  s  ON r.sample_id   = s.sample_id
        WHERE gc.filter_status = 'PASS'
          AND v.alt_allele != '.'
    """)
    print(f'DB ALT variants loaded: {len(db_df):,} rows')

    # Group by run_id
    db_vars  = {}
    db_labels = {}
    for row in db_df.itertuples():
        db_vars.setdefault(row.run_id, set()).add(
            (row.chromosome, row.position, row.ref_allele, row.alt_allele)
        )
        db_labels[row.run_id] = row.label

    # Compute overlap
    results = []
    for rid, svars in db_vars.items():
        shared = len(query_variants & svars)
        overlap_pct = 100 * shared / len(query_variants) if query_variants else 0
        results.append({'run_id': rid, 'label': db_labels.get(rid, rid),
                        'overlap_pct': overlap_pct, 'shared': shared,
                        'db_total': len(svars),
                        'is_self': rid == QUERY_RUN})
    res_df = pd.DataFrame(results).sort_values('overlap_pct', ascending=False)

    print('\nTop 10 matches:')
    display(res_df.head(10)[['run_id','label','overlap_pct','shared','db_total']].round(2))
    print(f'\nNote: self-match ({QUERY_RUN}) is included — this is expected to score ~100%.')
    print(f'In a real held-out test, remove the run from the DB before matching.')

    # Figure: bar chart sorted by overlap
    bar_colors = [
        C['same_donor'] if r == QUERY_RUN else
        (C['wes'] if '_wes_' in r.lower() else C['accent_teal'])
        for r in res_df['run_id']
    ]

    fig9 = go.Figure(go.Bar(
        x=res_df['overlap_pct'], y=res_df['label'],
        orientation='h',
        marker_color=bar_colors,
        hovertemplate='%{y}<br>Overlap: %{x:.1f}%<extra></extra>',
    ))
    fig9.add_vline(x=40, line_dash='dash', line_color=C['dark_teal'],
                  annotation_text='40% reporting threshold',
                  annotation_position='top')
    _style(fig9, showlegend=False)
    fig9.update_xaxes(title_text='Query overlap (%)', range=[0, 105])
    fig9.update_yaxes(title_text='Reference sample', tickfont=dict(size=12))
    # Annotate self-match
    self_row = res_df[res_df['run_id'] == QUERY_RUN]
    if not self_row.empty:
        fig9.add_annotation(
            x=self_row['overlap_pct'].iloc[0] + 1,
            y=self_row['label'].iloc[0],
            text='← self-match (RKO)',
            showarrow=False, xanchor='left',
            font=dict(size=14, color=C['same_donor']),
        )
    h9 = max(500, 18 * len(res_df))
    save(fig9, 'fig9_sample_identification', h=h9)

Query VCF (results/SRR5071686_hg38/vcf/SRR5071686.panel_genotyped.vcf): 11,639 ALT variants


DB ALT variants loaded: 3,050,552 rows



Top 10 matches:


,run_id,label,overlap_pct,shared,db_total
40,SRR5071686_hg38,RKO,100.00,11639,11639
38,SRR5071667_hg38,HCT116,44.34,5161,15783
37,SRR5071662_hg38,DLD1,43.86,5105,17960
41,SRR5071691_hg38,SW480,43.64,5079,21162
24,SRR5071669_hg38,HCT8,43.47,5060,16389
3,SRR5071661_hg38,DiFi,43.18,5026,20450
6,SRR5071674_hg38,IS1,42.65,4964,18327
8,SRR5071681_hg38,LIM2405,42.32,4926,15940
19,SRR5071660_hg38,COLO320,41.08,4781,15992
0,SRR5071656_hg38,C70,40.60,4725,15925



Note: self-match (SRR5071686_hg38) is included — this is expected to score ~100%.
In a real held-out test, remove the run from the DB before matching.


  ✓ figures/presentation/fig9_sample_identification.png / .svg


## Figure 10 — WES organoid Jaccard heatmap (slide 11)
**Slide message:** All WES organoid samples are from unrelated donors — no two pairs reach same-donor similarity. IKPT/IKPM and NCO/NCO_A values are reported for context; similarity is not claimed as evidence of a shared donor.

In [12]:
if len(wes_runs) < 2:
    print('Fewer than 2 WES runs — skipping Fig 10.')
else:
    print('Loading WES variant keys...')
    wes_keys = {}
    wes_ids  = wes_runs['run_id'].tolist()
    wes_lbl  = wes_runs['label'].tolist()
    for rid in wes_ids:
        df = query("""
            SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS k
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id=? AND g.filter_status='PASS'
        """, (rid,))
        wes_keys[rid] = set(df['k'].tolist())
    print('  Done.')

    m = len(wes_ids)
    wes_jac = np.zeros((m, m))
    for i, ri in enumerate(wes_ids):
        for j, rj in enumerate(wes_ids):
            si, sj = wes_keys[ri], wes_keys[rj]
            u = len(si | sj)
            wes_jac[i, j] = len(si & sj) / u if u else 0

    # Report IKPT/IKPM and NCO/NCO_A values
    pairs_of_interest = [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]
    triu = np.triu_indices(m, k=1)
    all_off = wes_jac[triu]
    med_bg = float(np.median(all_off))
    print(f'Median background Jaccard (all unrelated pairs): {med_bg:.4f}')
    for la, lb in pairs_of_interest:
        if la in wes_lbl and lb in wes_lbl:
            ia, ib = wes_lbl.index(la), wes_lbl.index(lb)
            v = wes_jac[ia, ib]
            print(f'{la} vs {lb}: Jaccard = {v:.4f}  (background median {med_bg:.4f})')
        else:
            print(f'{la} vs {lb}: one or both not in WES runs')

    # Highlight cells with shapes
    shapes10 = []
    highlight_pairs = [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]
    highlight_colors = [C['wes'], C['same_donor']]
    for (la, lb), hcolor in zip(highlight_pairs, highlight_colors):
        if la in wes_lbl and lb in wes_lbl:
            for pa, pb in [(la, lb), (lb, la)]:
                xi, yi = wes_lbl.index(pb), wes_lbl.index(pa)
                shapes10.append(dict(
                    type='rect', x0=xi-0.5, x1=xi+0.5, y0=yi-0.5, y1=yi+0.5,
                    line=dict(color=hcolor, width=3),
                    fillcolor='rgba(0,0,0,0)',
                    xref='x', yref='y',
                ))

    fig10 = go.Figure(go.Heatmap(
        z=wes_jac, x=wes_lbl, y=wes_lbl,
        colorscale='Purples', zmin=0, zmax=1,
        text=np.round(wes_jac, 3),
        texttemplate='%{text}',
        textfont=dict(size=13),
        hovertemplate='%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>',
        showscale=True,
        colorbar=dict(len=0.8, title='Jaccard', tickfont=dict(size=14)),
    ))
    _style(fig10, showlegend=False)
    fig10.update_layout(
        shapes=shapes10,
        annotations=[
            dict(x=0.02, y=-0.10, xref='paper', yref='paper',
                 text='Purple outline: IKPT/IKPM  |  Orange outline: NCO/NCO_A',
                 showarrow=False, font=dict(size=13, color='#555555'), xanchor='left'),
        ],
    )
    fig10.update_xaxes(tickangle=45, showgrid=False)
    fig10.update_yaxes(showgrid=False, autorange='reversed')
    sz = max(600, 60 * m)
    save(fig10, 'fig10_wes_jaccard_heatmap', w=sz, h=sz)

Loading WES variant keys...


  Done.


Median background Jaccard (all unrelated pairs): 0.2182
IKPT vs IKPM: Jaccard = 0.3116  (background median 0.2182)
NCO vs NCO_A: Jaccard = 0.3212  (background median 0.2182)


  ✓ figures/presentation/fig10_wes_jaccard_heatmap.png / .svg


## numbers.md — Slide numbers
Every number intended for a slide, with its source query or file.

In [13]:
from datetime import date
sc_src = "SELECT count(*) FROM runs JOIN samples ... WHERE run_id NOT LIKE '%_wes_%'"
lines = [
    f'# Slide numbers — generated {date.today()}',
    '',
    '## Cohort overview',
    f'- sc/bulk runs in DB: **{len(sc_runs)}** (source: runs table, dtype != WES)',
    f'- WES runs in DB: **{len(wes_runs)}**',
    f'- Total runs: **{len(all_runs)}**',
    '',
]
# QC numbers
lines += [
    '## QC (Fig 5)',
    f'- sc/bulk mapping rates: {sc_runs["mapping_rate"].min():.1f}% – {sc_runs["mapping_rate"].max():.1f}%  (source: runs.mapping_rate)',
    f'- WES mapping rates:     {wes_runs["mapping_rate"].min():.2f}% – {wes_runs["mapping_rate"].max():.2f}%',
    '- Pipeline abort threshold: 85% (source: config.yaml samtools.min_mapping_rate)',
    '',
]

# Same-donor distribution numbers
if len(jac_same) > 0:
    lines += [
        '## Same-donor vs different-donor (Fig 3)',
        f'- Same-donor Jaccard pairs: **{len(jac_same)}** (8 biological pairs)',
        f'- Same-donor Jaccard median: **{np.median(jac_same):.4f}**  (source: pairwise sc_jac matrix)',
        f'- Diff-donor Jaccard median: **{np.median(jac_diff):.4f}**',
        f'- Min same-donor Jaccard: {jac_same.min():.4f}',
        f'- Max diff-donor Jaccard: {jac_diff.max():.4f}',
        f'- Diff-donor pairs above same-donor min: {(jac_diff >= jac_same.min()).sum()}',
        f'- Same-donor containment median: **{np.median(con_same):.4f}**',
        f'- Diff-donor containment median: **{np.median(con_diff):.4f}**',
        '',
    ]

# Pool_ctr numbers
if fa_g is not None:
    SINGLET_SRC = {'db_match', 'guided_match'}
    total_g = len(fa_g)
    sing_g  = fa_g[fa_g['source'].isin(SINGLET_SRC)]
    lines += [
        '## Pool_ctr cell assignment (Fig 6)',
        f'- Guided run: {GUIDED_RUN}   total cells: **{total_g:,}** (source: final_assignments.tsv)',
        f'- Guided singlets: **{100*len(sing_g)/total_g:.1f}%**',
        f'- Guided doublets: **{100*(fa_g["source"]=="doublet").sum()/total_g:.1f}%**',
        f'- Guided unassigned: **{100*(fa_g["source"]=="unassigned").sum()/total_g:.1f}%**',
        f'- High-confidence cells: **{100*fa_g["match_confidence"].isin(["high","guided"]).sum()/total_g:.1f}%**',
    ]
    if fa_u is not None:
        total_u = len(fa_u)
        sing_u = fa_u[fa_u['source'].isin(SINGLET_SRC)]
        lines += [
            f'- Unguided run: {UNGUIDED_RUN}   total cells: **{total_u:,}**',
            f'- Unguided singlets: **{100*len(sing_u)/total_u:.1f}%** (only 3/7 lines identified)',
        ]
    lines.append('')

# Donor matching
if 'dm' in dir():
    high_d = dm[dm['confidence'] == 'high']
    best_idx = high_d['concordance'].argmax()
    best_line = high_d.iloc[best_idx]['cell_line']
    best_conc = high_d['concordance'].max()
    lines += [
        '## Donor matching (Fig 8)',
        f'- Donors matched with high confidence: {len(high_d)}/7',
        f'- Max concordance (best match): {best_conc:.4f}  ({best_line})',
        '- Threshold used (inferred): min_concordance=0.80, min_gap=0.15',
        '  source: pool_ctr_v4/donor_matches.tsv — post-backfill thresholds',
        '',
    ]

# WES
if 'wes_jac' in dir() and len(wes_runs) >= 2:
    triu = np.triu_indices(len(wes_ids), k=1)
    wes_off = wes_jac[triu]
    lines += [
        '## WES similarity (Fig 10)',
        f'- WES all-pairs Jaccard median: **{np.median(wes_off):.4f}**  (source: pairwise wes_jac matrix)',
        f'- WES all-pairs Jaccard range: {wes_off.min():.4f} – {wes_off.max():.4f}',
    ]
    for la, lb in [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]:
        if la in wes_lbl and lb in wes_lbl:
            ia, ib = wes_lbl.index(la), wes_lbl.index(lb)
            lines.append(f'- {la} vs {lb} Jaccard: **{wes_jac[ia,ib]:.4f}** (vs background median {np.median(wes_off):.4f})')
    lines.append('')

out_path = f'{FIG_DIR}/numbers.md'
with open(out_path, 'w') as f:
    f.write('\n'.join(lines))
print(f'Written: {out_path}')

# List all generated files
print('\n=== Generated files ===')
all_files = sorted(glob.glob(f'{FIG_DIR}/**/*', recursive=True) +
                   glob.glob(f'{FIG_DIR}/*'))
for fp in sorted(set(all_files)):
    if os.path.isfile(fp):
        size = os.path.getsize(fp)
        unit = 'KB' if size < 1e6 else 'MB'
        val  = size / 1e3 if size < 1e6 else size / 1e6
        print(f'  {fp}  ({val:.0f} {unit})')

Written: figures/presentation/numbers.md

=== Generated files ===
  figures/presentation/fig10_wes_jaccard_heatmap.png  (248 KB)
  figures/presentation/fig10_wes_jaccard_heatmap.svg  (34 KB)
  figures/presentation/fig2_zero_zero_schematic.png  (213 KB)
  figures/presentation/fig2_zero_zero_schematic.svg  (20 KB)
  figures/presentation/fig3_same_vs_diff_donor.png  (484 KB)
  figures/presentation/fig3_same_vs_diff_donor.svg  (537 KB)
  figures/presentation/fig4_sc_jaccard_heatmap.png  (248 KB)
  figures/presentation/fig4_sc_jaccard_heatmap.svg  (32 KB)
  figures/presentation/fig5_qc_summary.png  (509 KB)
  figures/presentation/fig5_qc_summary.svg  (53 KB)
  figures/presentation/fig6_pool_ctr_composition.png  (309 KB)
  figures/presentation/fig6_pool_ctr_composition.svg  (29 KB)
  figures/presentation/fig7_guided_vs_unguided.png  (671 KB)
  figures/presentation/fig7_guided_vs_unguided.svg  (162 KB)
  figures/presentation/fig8_donor_matching.png  (327 KB)
  figures/presentation/fig8_donor_